In [22]:
import importlib
import services.agent
importlib.reload(services.agent)
print("Reloaded services.agent module")

Reloaded services.agent module


# Phase 6 Testing with Real Data

This notebook demonstrates end-to-end testing of Phase 6 using:
1. OCR extraction from sample_table1.png
2. Classifier predictions
3. RAG retrieval of relevant chunks
4. LLM-based analysis

We'll process actual patient data through the entire pipeline.

In [23]:
try:
    # First reload the modules to clear cache
    import importlib
    import services.agent
    import services.parser
    importlib.reload(services.agent)
    importlib.reload(services.parser)
    print("Reloaded modules")
    
    # Now try the imports
    from services.classifier.api.main import predict_risk
    from services.rag.api import search_guidelines
    from services.agent import DiabetesAgent
    from services.parser.parser import MedicalTableParser
    print("Successfully imported all modules")
except ImportError as e:
    print(f"Import error: {e}")
    print("\nTrying to debug module contents...")
    import services.agent
    print("Contents of services.agent:", dir(services.agent))

Reloaded modules
Successfully imported all modules


## 1. OCR Processing

First, we'll use Tesseract OCR to extract text from the sample table image. We'll need to:
1. Load and preprocess the image
2. Run OCR
3. Parse the extracted text into structured data

In [24]:
from pathlib import Path
import json

# Notebook dir = where the .ipynb is
notebook_dir = Path.cwd()

# Go up to project root (one level up from notebooks)
project_root = notebook_dir.parent  

data_dir = project_root / "data"
sample_image_path = data_dir / "raw" / "sample_table1.png"

print(f"Using image path: {sample_image_path}")
if not sample_image_path.exists():
    raise FileNotFoundError(f"Image not found at {sample_image_path}")

# Initialize the parser
parser = MedicalTableParser(use_advanced_model=False)  # Using Tesseract by default

# Parse the table from image
patient_data = parser.parse_image(str(sample_image_path))

print("\nExtracted patient data:")
print("-" * 40)
print(json.dumps(patient_data, indent=2))


Using image path: C:\Users\prane\Desktop\Project 1\Code\data\raw\sample_table1.png

===== OCR OUTPUT FOR sample_table1.png =====
Pregnancies Glucose BloodPressure SkinThickness Insulin BMI DiabetesPedigreeFunction Age Outcome
6 148 72 35 fe} 33.6 0.627 50 1

===== END OCR OUTPUT =====


===== PARSED DATAFRAME =====
  Pregnancies Glucose BloodPressure SkinThickness Insulin   BMI  \
0           6     148            72            35     fe}  33.6   

  DiabetesPedigreeFunction Age Outcome  
0                    0.627  50       1  


Extracted patient data:
----------------------------------------
{
  "Pregnancies": 6.0,
  "Glucose": 148.0,
  "BloodPressure": 72.0,
  "SkinThickness": 35.0,
  "Insulin": null,
  "BMI": 33.6,
  "DiabetesPedigreeFunction": 0.627,
  "Age": 50.0,
  "Outcome": 1.0
}


## 2. Get Classifier Predictions

Now that we have structured patient data, let's get predictions from our classifier API. We'll:
1. Format the data for the classifier
2. Call the prediction endpoint
3. Process the results

In [25]:
import asyncio

# Get classifier prediction
async def get_prediction(patient_data: dict):
    """Get prediction from classifier API."""
    try:
        result = await predict_risk(patient_data)
        return result
    except Exception as e:
        print(f"Error getting prediction: {e}")
        return None

# Run prediction
classifier_output = await get_prediction(patient_data)
print("Classifier output:")
print(json.dumps(classifier_output, indent=2))

Classifier output:
{
  "prediction": "low_risk",
  "probability": 0.0012665705289691687,
  "top_features": [
    "Glucose",
    "BMI",
    "DiabetesPedigreeFunction"
  ]
}


## 3. Retrieve Relevant Guidelines

Using the patient data and classifier output, we'll now retrieve relevant medical guidelines:
1. Construct a search query from patient data
2. Call the RAG API to get relevant chunks
3. Process the retrieved chunks

In [28]:
# Build search query from patient data
def build_search_query(patient_data: dict, classifier_output: dict) -> str:
    """Build a search query based on patient data and classifier output."""
    risk_level = classifier_output["prediction"]
    
    query_parts = [
        f"guidelines for {risk_level} diabetes patient",
        f"BMI {patient_data.get('BMI', 'N/A')}",
        f"blood pressure {patient_data.get('BloodPressure', 'N/A')}",
        f"glucose {patient_data.get('Glucose', 'N/A')}"
    ]
    
    return " ".join(query_parts)

# Get relevant chunks
async def get_relevant_chunks(query: str):
    """Get relevant chunks from RAG API."""
    try:
        chunks = await search_guidelines(query)
        return chunks
    except Exception as e:
        print(f"Error retrieving chunks: {e}")
        return []

# Run search
query = build_search_query(patient_data, classifier_output)
print(f"Search query: {query}")

retrieved_chunks = await get_relevant_chunks(query)
print(f"\nRetrieved {len(retrieved_chunks)} chunks:")
for i, chunk in enumerate(retrieved_chunks[:3], 1):  # Show first 3 chunks
    print(f"\nChunk {i}:")
    print(f"Document: {chunk['doc_id']}, Page: {chunk['page_num']}")
    print(f"Text: {chunk['text'][:200]}...")

Search query: guidelines for low_risk diabetes patient BMI 33.6 blood pressure 72.0 glucose 148.0

Retrieved 5 chunks:

Chunk 1:
Document: WHO_Fact_Sheet, Page: 7
Text: Along with medicines to lower blood sugar, people with diabetes often need medications to lower their blood pressure and statins to reduce the risk of complications. Additional medical care may be nee...

Chunk 2:
Document: WHO_Fact_Sheet, Page: 2
Text: Diabetes can be treated and its consequences avoided or delayed with diet, physical activity, medication and regular screening and treatment for complications. Diabetes is a chronic disease that occur...

Chunk 3:
Document: IDF_Atlas, Page: 3
Text: 589 million adults (20-79 years) are living with diabetes – 1 in 9. This number is predicted to rise to 853 Diabetes was responsible for 3.4 million deaths in 2024 – 1 every 9 Diabetes caused at least...


## 4. Direct Groq LLM Analysis

Let's use Groq LLM directly to analyze our data:
1. Set up Groq client
2. Format our data
3. Get LLM analysis with citations

In [29]:
import groq

# Initialize Groq client
if "GROQ_API_KEY" not in os.environ:
    api_key = input("Enter your Groq API key: ")
    os.environ["GROQ_API_KEY"] = api_key
    
groq_client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])

# Format our data for the prompt
context = f"""
Patient Data:
{json.dumps(patient_data, indent=2)}

Classifier Output:
{json.dumps(classifier_output, indent=2)}

Retrieved Medical Guidelines:
"""

for i, chunk in enumerate(retrieved_chunks, 1):
    context += f"\nChunk {i}:\n"
    context += f"Document: {chunk['doc_id']}, Page: {chunk['page_num']}\n"
    context += f"Content: {chunk['text']}\n"

system_prompt = """You are a medical analysis assistant. Analyze the patient data, classifier output, and medical guidelines provided.
Your response must be in JSON format with the following structure:
{
    "conclusion": "Clear summary of patient's condition and risk factors",
    "reasoning": ["Step 1 of analysis", "Step 2", "etc"],
    "next_steps": ["Recommended action 1", "Action 2", "etc"],
    "sources": [
        {
            "doc_id": "Source document ID",
            "page": page_number,
            "quote": "Exact quote from source",
            "relevance": "Why this source supports the analysis"
        }
    ]
}

Always cite specific guidelines and include exact quotes when making recommendations."""

# Call Groq
chat_completion = groq_client.chat.completions.create(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": context}
    ],
    model="llama-3.1-8b-instant",  # Using Mixtral model
    temperature=0.3,
    max_tokens=2048
)

# Parse and display results
result = json.loads(chat_completion.choices[0].message.content)

print("Groq Analysis:")
print("-" * 40)
print(f"Conclusion: {result['conclusion']}\n")

print("Reasoning:")
for i, step in enumerate(result['reasoning'], 1):
    print(f"{i}. {step}")
print()

print("Next Steps:")
for i, step in enumerate(result['next_steps'], 1):
    print(f"{i}. {step}")
print()

print("Sources:")
for i, source in enumerate(result['sources'], 1):
    print(f"\nSource {i}:")
    print(f"Document: {source['doc_id']}, Page: {source['page']}")
    print(f"Quote: {source['quote']}")
    print(f"Relevance: {source['relevance']}")

Groq Analysis:
----------------------------------------
Conclusion: The patient is at a high risk of developing diabetes due to their high glucose levels, BMI, and family history. Immediate action is required to prevent or delay the onset of type 2 diabetes.

Reasoning:
1. The patient's glucose level of 148.0 is higher than the normal range, indicating insulin resistance or impaired glucose regulation.
2. The patient's BMI of 33.6 is classified as obese, which is a significant risk factor for developing type 2 diabetes.
3. The patient's family history, as indicated by the DiabetesPedigreeFunction of 0.627, suggests a genetic predisposition to diabetes.
4. The patient's age of 50 is also a risk factor, as the risk of developing type 2 diabetes increases with age.

Next Steps:
1. Recommend a healthy diet, regular physical activity, and weight loss to prevent or delay the onset of type 2 diabetes.
2. Monitor the patient's glucose levels and BMI regularly to track their progress.
3. Consid